# Step 1: Design Schema & Generate Data

Uses an LLM to design a database schema based on the company description,
then Faker generates realistic data. Outputs are written to a UC Volume
for the next pipeline step.

In [ ]:
%pip install faker openai
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "yd_launchpad_final_classic_catalog")
dbutils.widgets.text("schema", "genie_app")
dbutils.widgets.text("company_name", "NovaTech Logistics")
dbutils.widgets.text("company_description", "NovaTech Logistics is a mid-size freight and logistics company operating across North America and Europe. They manage a fleet of 2,000+ trucks and work with 500+ warehouses. Their core data includes shipment tracking, warehouse inventory, fleet maintenance, driver performance, and customer contracts.")
dbutils.widgets.text("databricks_host_id", "7474655921234161")
dbutils.widgets.text("llm_model", "opendoor-claude-opus-46")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
COMPANY_NAME = dbutils.widgets.get("company_name")
COMPANY_DESC = dbutils.widgets.get("company_description")
DATABRICKS_HOST_ID = dbutils.widgets.get("databricks_host_id")
LLM_MODEL = dbutils.widgets.get("llm_model")

import re
def slugify(name: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', name.lower()).strip('_')
    if slug and slug[0].isdigit():
        slug = f'co_{slug}'
    return slug[:50]

COMPANY_SLUG = slugify(COMPANY_NAME)
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{COMPANY_SLUG}"

print(f"Company: {COMPANY_NAME}")
print(f"Slug: {COMPANY_SLUG}")
print(f"All tables go in: {CATALOG}.{SCHEMA}")
print(f"Volume: {VOLUME_PATH}")

In [ ]:
# Ensure the volume subdirectory exists (schema + volume already exist)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.raw_data")
# Volume subdirs are created automatically when writing files
print(f"Schema and volume ready: {CATALOG}.{SCHEMA}")

In [ ]:
import json
from openai import OpenAI

# Get token from notebook context
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()

client = OpenAI(
    api_key=token,
    base_url=f"https://{DATABRICKS_HOST_ID}.ai-gateway.cloud.databricks.com/mlflow/v1",
)

SYSTEM_PROMPT = """You are a data architect. Given a company description, design a realistic database schema.
Output ONLY valid JSON, no markdown fences, no trailing commas.

RULES:
1. Create 3-5 tables that represent the company's core business entities and transactions.
2. Each table should have 5-12 columns with realistic names and types.
3. Every table MUST have a primary key column as the first column, using "sequential_id" faker provider.
4. Use foreign keys ("fk" provider) to create relationships between tables.
5. Include at least one date column and one numeric/financial column per table.
6. Use "random_element" for categorical columns with domain-specific values (5-10 realistic options).
7. Make column names snake_case and table names snake_case plural (e.g., "shipments", "warehouses").
8. Add meaningful table comments and column comments.
9. Include sample_questions (5-7) that a business user would ask about this data.
10. Tables should be ordered so that referenced tables come before tables that reference them.

AVAILABLE FAKER PROVIDERS:
  - "sequential_id"    -> auto-increment integer (1, 2, 3, ...). SQL type: INT.
  - "name"             -> full name. SQL type: STRING.
  - "first_name"       -> first name. SQL type: STRING.
  - "last_name"        -> last name. SQL type: STRING.
  - "email"            -> email address. SQL type: STRING.
  - "phone_number"     -> phone number. SQL type: STRING.
  - "job"              -> job title. SQL type: STRING.
  - "company"          -> company name. SQL type: STRING.
  - "catch_phrase"     -> marketing phrase. SQL type: STRING.
  - "address"          -> full street address. SQL type: STRING.
  - "city"             -> city name. SQL type: STRING.
  - "state"            -> state/province. SQL type: STRING.
  - "country"          -> country name. SQL type: STRING.
  - "zipcode"          -> postal code. SQL type: STRING.
  - "latitude"         -> latitude float. SQL type: DOUBLE.
  - "longitude"        -> longitude float. SQL type: DOUBLE.
  - "date_between"     -> random date. Requires "args": {"start_date": "-2y", "end_date": "today"}. SQL type: DATE.
  - "date_this_year"   -> date in current year. SQL type: DATE.
  - "random_int"       -> random integer. Requires "args": {"min": 1, "max": 100}. SQL type: INT.
  - "pyfloat"          -> random float. Requires "args": {"min_value": 0, "max_value": 1000, "right_digits": 2}. SQL type: DOUBLE.
  - "random_element"   -> pick from a list. Requires "args": {"elements": ["A", "B", "C"]}. SQL type: STRING.
  - "boolean"          -> true/false. Optional "args": {"chance_of_getting_true": 75}. SQL type: BOOLEAN.
  - "text"             -> paragraph. Optional "args": {"max_nb_chars": 200}. SQL type: STRING.
  - "sentence"         -> single sentence. SQL type: STRING.
  - "uuid4"            -> UUID string. SQL type: STRING.
  - "url"              -> URL. SQL type: STRING.
  - "currency_code"    -> 3-letter currency code. SQL type: STRING.
  - "fk"               -> foreign key. Requires "args": {"references": "table_name.column_name"}. SQL type: INT.

OUTPUT FORMAT (strict JSON):
{
  "tables": [
    {
      "name": "table_name",
      "comment": "Description",
      "row_count": 1000,
      "columns": [
        {"name": "col_name", "faker": "provider_name", "args": {}, "comment": "Description"}
      ]
    }
  ],
  "sample_questions": ["Question 1?", "Question 2?"]
}"""

print("Calling LLM to design schema...")
resp = client.chat.completions.create(
    model=LLM_MODEL,
    max_tokens=8192,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": COMPANY_DESC},
    ],
)

raw = resp.choices[0].message.content.strip()
# Clean up LLM output
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1]
    if raw.endswith("```"):
        raw = raw[:raw.rfind("```")]
start = raw.find("{")
end = raw.rfind("}") + 1
if start >= 0 and end > start:
    raw = raw[start:end]
raw = re.sub(r",\s*([}\]])", r"\1", raw)

schema = json.loads(raw)
print(f"Schema designed: {len(schema['tables'])} tables, {len(schema.get('sample_questions', []))} sample questions")
for t in schema["tables"]:
    print(f"  - {t['name']} ({t.get('row_count', 0)} rows, {len(t['columns'])} columns)")

In [ ]:
from faker import Faker
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, BooleanType
import datetime

fake = Faker()
Faker.seed(42)

PROVIDER_SPARK_TYPES = {
    "sequential_id": IntegerType(),
    "name": StringType(), "first_name": StringType(), "last_name": StringType(),
    "email": StringType(), "phone_number": StringType(), "job": StringType(),
    "company": StringType(), "catch_phrase": StringType(), "bs": StringType(),
    "address": StringType(), "city": StringType(), "state": StringType(),
    "country": StringType(), "zipcode": StringType(),
    "latitude": DoubleType(), "longitude": DoubleType(),
    "date_between": DateType(), "date_this_year": DateType(),
    "random_int": IntegerType(),
    "pyfloat": DoubleType(),
    "random_element": StringType(),
    "boolean": BooleanType(),
    "text": StringType(), "sentence": StringType(),
    "uuid4": StringType(), "url": StringType(), "currency_code": StringType(),
    "fk": IntegerType(),
}

def generate_value(provider, args, row_idx, generated_tables):
    if provider == "sequential_id":
        return row_idx + 1
    if provider == "fk":
        ref = args.get("references", "")
        ref_table = ref.split(".")[0] if "." in ref else ref
        ref_col = ref.split(".")[1] if "." in ref else f"{ref}_id"
        ref_data = generated_tables.get(ref_table, [])
        if ref_data:
            return fake.random_element(ref_data).get(ref_col, 1)
        return fake.random_int(min=1, max=100)
    if provider == "random_element":
        return fake.random_element(args.get("elements", ["A", "B", "C"]))
    if provider == "random_int":
        return fake.random_int(min=args.get("min", 1), max=args.get("max", 100))
    if provider == "pyfloat":
        return round(fake.pyfloat(min_value=args.get("min_value", 0), max_value=args.get("max_value", 1000), right_digits=args.get("right_digits", 2)), args.get("right_digits", 2))
    if provider == "date_between":
        return fake.date_between(start_date=args.get("start_date", "-2y"), end_date=args.get("end_date", "today"))
    if provider == "boolean":
        return fake.boolean(chance_of_getting_true=args.get("chance_of_getting_true", 50))
    if provider == "text":
        return fake.text(max_nb_chars=args.get("max_nb_chars", 200))
    faker_fn = getattr(fake, provider, None)
    if faker_fn and callable(faker_fn):
        return faker_fn()
    return fake.sentence()

def coerce_value(val, spark_type):
    """Force values to the correct Python type for Spark."""
    if val is None:
        return None
    if isinstance(spark_type, IntegerType):
        return int(val)
    if isinstance(spark_type, DoubleType):
        return float(val)
    if isinstance(spark_type, BooleanType):
        return bool(val)
    if isinstance(spark_type, DateType):
        if isinstance(val, datetime.date):
            return val
        return datetime.date.fromisoformat(str(val))
    return str(val)

# Generate all tables and write as parquet
generated_tables = {}

for table_def in schema["tables"]:
    table_name = table_def["name"]
    row_count = table_def.get("row_count", 1000)
    columns = table_def["columns"]
    
    print(f"\nGenerating {row_count} rows for '{table_name}'...")
    
    # Generate rows
    rows = []
    for i in range(row_count):
        row = {}
        for col in columns:
            row[col["name"]] = generate_value(col["faker"], col.get("args", {}), i, generated_tables)
        rows.append(row)
    
    generated_tables[table_name] = rows
    
    # Build Spark schema and coerce types
    fields = []
    for col in columns:
        spark_type = PROVIDER_SPARK_TYPES.get(col["faker"], StringType())
        fields.append(StructField(col["name"], spark_type, True))
    spark_schema = StructType(fields)
    
    # Coerce every value to the right Python type
    clean_rows = []
    for row in rows:
        clean_row = {}
        for col, field in zip(columns, fields):
            clean_row[col["name"]] = coerce_value(row[col["name"]], field.dataType)
        clean_rows.append(clean_row)
    
    # Prefix table name with company slug for storage
    prefixed_name = f"{COMPANY_SLUG}_{table_name}"
    df = spark.createDataFrame(clean_rows, spark_schema)
    output_path = f"{VOLUME_PATH}/{table_name}/"
    df.write.mode("overwrite").parquet(output_path)
    print(f"  Written {row_count} rows to {output_path}")

print(f"\nAll {len(schema['tables'])} tables generated!")

In [ ]:
# Save the schema definition + metadata for the next pipeline steps
pipeline_state = {
    "company_name": COMPANY_NAME,
    "company_description": COMPANY_DESC,
    "company_slug": COMPANY_SLUG,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "tables": schema["tables"],
    "sample_questions": schema.get("sample_questions", []),
}

state_json = json.dumps(pipeline_state, indent=2)
dbutils.fs.put(f"{VOLUME_PATH}/pipeline_state.json", state_json, overwrite=True)

print(f"Pipeline state saved to {VOLUME_PATH}/pipeline_state.json")
print(f"\nSample questions:")
for q in schema.get("sample_questions", []):
    print(f"  - {q}")